# Optional Project - Colab Part1 (No Drive)

This notebook runs Task1 only: preprocess, tokenizer, and HF publishing.


In [1]:
# ===== User config =====
REPO_URL = "https://github.com/Peng-y-x/optionalproject.git"
REPO_DIR = "/content/optionalproject"
REPO_BRANCH = "run"
DATA_CONFIG = "configs/data.yaml"
# DATA_CONFIG = "configs/data_vocab1024.yaml"
# DATA_CONFIG = "configs/data_vocab8192.yaml"


In [8]:
# 1) Clone repo and checkout branch
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists:', REPO_DIR)
%cd $REPO_DIR
!git fetch origin
!git checkout {REPO_BRANCH}
!git pull origin {REPO_BRANCH}
!git branch --show-current
!git rev-parse --short HEAD


Repo already exists: /content/optionalproject
/content/optionalproject
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 363 bytes | 363.00 KiB/s, done.
From https://github.com/Peng-y-x/optionalproject
   fe8ae37..d669e5b  run        -> origin/run
Already on 'run'
Your branch is behind 'origin/run' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/Peng-y-x/optionalproject
 * branch            run        -> FETCH_HEAD
Updating fe8ae37..d669e5b
Fast-forward
 configs/data.yaml | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)
run
d669e5b


In [9]:
# 2) Install system + Python dependencies
!apt-get update -y
!apt-get install -y libcairo2 libcairo2-dev libffi-dev
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -U datasets huggingface_hub cairosvg tokenizers


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease                         
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease          
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.ill

In [6]:
# 3) HF auth from Colab Keys (key name must be HF_TOKEN)
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
    print('HF token loaded from Colab key.')
else:
    print('HF token not found in Colab key HF_TOKEN.')
print('has_hf_token:', bool(os.getenv('HF_TOKEN')))


TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [11]:
# 4) Render sanity check (must be True before preprocess)
from src.data.validate_svg import validate_render
svg = '<svg xmlns="http://www.w3.org/2000/svg" width="24" height="24"><circle cx="12" cy="12" r="6"/></svg>'
ok, err = validate_render(svg)
print('render_check_ok:', ok)
print('render_check_err:', err)


render_check_ok: True
render_check_err: None


In [13]:
# 5) Preprocess data (and push clean dataset if hf_push.enabled=true)
%cd $REPO_DIR
# !python scripts/run_preprocess.py --config {DATA_CONFIG}
!python scripts/run_preprocess.py --config {DATA_CONFIG} --force

/content/optionalproject
[auth] Loaded HF token from Colab key.
[1/8] Checking processed cache + manifest...
  Force rebuild enabled. Regenerating outputs.
[2/8] Loading source datasets from Hugging Face (uses cache_dir for reuse)...
  Loaded raw records: 172500
[3/8] Cleaning + filtering + validation...
  Records after cleaning + filtering: 170063
  Dropped summary: {"empty_after_clean": 0, "too_short": 0, "too_long_chars": 582, "too_long_tokens_est": 0, "invalid_xml": 0, "non_svg_root": 0, "render_failed": 0, "dedup_svg_hash": 1855}
[4/8] Building train/validation/test split...
[5/8] Writing stats + histograms...
[6/8] Exporting complexity examples (SVG + PNG)...
[7/8] Optional push to HF hub...
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Uploading the dataset shards:   0% 0/1 [00:00<?, ? shards/s]
Creating parquet from Arrow format:   0% 0/3 [00:00<?, ?ba/s]
Creating parquet from Arrow format:  33% 1/3 [00:00<00:00,

In [ ]:
# 5) or download preprocessed data from huggingface
from datasets import load_dataset
import json, os

repo_id = "Zala0429/svg-scaling-v1-clean"
out_dir = "data/processed/v1-clean-rawsplit"
os.makedirs(out_dir, exist_ok=True)

ds = load_dataset(repo_id)
for split, fname in [("train","train.jsonl"), ("validation","validation.jsonl"), ("test","test.jsonl")]:
    with open(f"{out_dir}/{fname}", "w", encoding="utf-8") as f:
        for row in ds[split]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


In [14]:
# 6) Train tokenizer + encode splits (enforces real train token target)
%cd $REPO_DIR
!python scripts/run_tokenizer.py --config {DATA_CONFIG}


/content/optionalproject
[1/4] Preparing tokenizer training text...
[2/4] Training BPE tokenizer...
[00:00:00] Tokenize words                 ██████████████████ 8272     /     8272[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:00] Count pairs                    ██████████████████ 8272     /     8272
[00:00:00] Compute merges                 ██████████████████ 4046     /     4046
  tokenizer: data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json
[3/4] Encoding train/validation/test splits...
[4/4] Writing tokenization stats...
Done.
Vocab size: 4096
Token totals:
  train: 100626064
  validation: 1008216
  test: 1041841


In [15]:
# 7) Push tokenizer artifacts to HF model repo
%cd $REPO_DIR
!python scripts/push_tokenizer_to_hf.py --config {DATA_CONFIG}


/content/optionalproject
[auth] Loaded HF token from Colab key.
No files have been modified since last commit. Skipping to prevent empty commit.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:10913: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")
No files have been modified since last commit. Skipping to prevent empty commit.
Pushed tokenizer artifacts to: Zala0429/svg-scaling-tokenizer-v1
{"repo_id": "Zala0429/svg-scaling-tokenizer-v1", "repo_type": "model"}


In [16]:
# 8) Push tokenized dataset to HF dataset repo
%cd $REPO_DIR
!python scripts/push_tokenized_dataset_to_hf.py --config {DATA_CONFIG}


/content/optionalproject
[auth] Loaded HF token from Colab key.
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Uploading the dataset shards:   0% 0/1 [00:00<?, ? shards/s]
Creating parquet from Arrow format:   0% 0/5 [00:00<?, ?ba/s]
Creating parquet from Arrow format:  20% 1/5 [00:00<00:02,  1.39ba/s]
Creating parquet from Arrow format:  40% 2/5 [00:01<00:02,  1.43ba/s]
Creating parquet from Arrow format:  60% 3/5 [00:02<00:01,  1.49ba/s]
Creating parquet from Arrow format:  80% 4/5 [00:02<00:00,  1.53ba/s]
Creating parquet from Arrow format: 100% 5/5 [00:02<00:00,  1.80ba/s]
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpzmwv1c6x.parquet    :   1% 2.22M/410M [00:00<?, ?B/s]

Processing Files (0 / 1)      :   1% 2.22M/410M [00:01<05:34, 1.22MB/s, 1.39MB/s  ]
New Data Upload               :   3% 2.22M/67.1M [00:01

In [17]:
# 9) Inspect key outputs (auto-resolve tokenizer output_dir from DATA_CONFIG)
import json
from pathlib import Path
import yaml

cfg = yaml.safe_load(Path(DATA_CONFIG).read_text(encoding='utf-8'))
root = Path(cfg['output']['dir'])
tokenizer_out_dir = Path(cfg['tokenization']['output_dir'])

stats = root / 'stats.json'
tok = tokenizer_out_dir / 'token_stats.json'
print('data_output_dir:', root)
print('tokenizer_output_dir:', tokenizer_out_dir)
print('stats exists:', stats.exists())
print('token_stats exists:', tok.exists())
if stats.exists():
    s = json.loads(stats.read_text(encoding='utf-8'))
    print('cleaned_records:', s.get('cleaned_records'))
    print('train_token_est_total:', s.get('train_token_est_total'))
if tok.exists():
    t = json.loads(tok.read_text(encoding='utf-8'))
    print('vocab_size:', t.get('vocab_size'))
    print('train_total_tokens:', t.get('splits', {}).get('train', {}).get('total_tokens'))


data_output_dir: data/processed/v1-clean-rawsplit
tokenizer_output_dir: data/processed/v1-clean-rawsplit/tokenizer
stats exists: True
token_stats exists: True
cleaned_records: 170063
train_token_est_total: 1229445
vocab_size: 4096
train_total_tokens: 100626064
